# ARGUS V3 — Domain-Adapted Retraining (Final, Leakage-Free)

## Purpose
Train an XGBoost model that understands YOUR Windows normal traffic.

## Training data
- **Benign:** Your 1,416 live Windows flows (first 80% = 1,132 flows)
- **Attack:** UNSW-NB15 attack rows (stratified random 80% = ~95K flows)

## Test data (never seen during training)
- **Benign:** Last 20% of your live Windows flows (284 flows, chronological)
- **Attack:** Stratified random 20% of UNSW-NB15 attacks (~24K flows)

## Why all live flows are treated as NORMAL
The 1,416 flows were collected during **controlled normal laptop usage**.
They are labeled NORMAL because of HOW they were collected,
NOT because the old model predicted them as NORMAL.
The old model predicted 1,353 of them as ATTACK — those predictions are discarded entirely.

## Cell 1 — Install dependencies

In [ ]:
!pip install xgboost scikit-learn pandas numpy -q
print('Ready')

## Cell 2 — Upload files

In [ ]:
from google.colab import files
print('Upload 1: UNSW_NB15_training-set.csv')
print('Upload 2: live_feature_capture.csv')
uploaded = files.upload()
print('Uploaded:', list(uploaded.keys()))

## Cell 3 — Feature definitions

In [ ]:
import pandas as pd
import numpy as np
import pickle, json, warnings
warnings.filterwarnings('ignore')

MODEL_FEATURE_ORDER = [
    'dur','proto','service','state','spkts','dpkts','sbytes','dbytes',
    'rate','sttl','dttl','sload','dload','sloss','dloss','sinpkt',
    'dinpkt','sjit','djit','swin','stcpb','dtcpb','dwin','tcprtt',
    'synack','ackdat','smean','dmean','trans_depth','response_body_len',
    'ct_srv_src','ct_state_ttl','ct_dst_ltm','ct_src_dport_ltm',
    'ct_dst_sport_ltm','ct_dst_src_ltm','is_ftp_login','ct_ftp_cmd',
    'ct_flw_http_mthd','ct_src_ltm','ct_srv_dst','is_sm_ips_ports',
]
CATEGORICAL = ['proto','service','state']
NUMERIC     = [f for f in MODEL_FEATURE_ORDER if f not in CATEGORICAL]
NON_FEATURES = ['prediction','confidence','flow_packets','timestamp','collected_at']

BENIGN_TRAIN_RATIO = 0.80
RANDOM_STATE       = 42

print(f'Model features: {len(MODEL_FEATURE_ORDER)}')
print(f'Categorical:    {CATEGORICAL}')
print(f'Numeric:        {len(NUMERIC)}')
print(f'Columns to drop (not model features): {NON_FEATURES}')

## Cell 4 — Load live Windows capture

In [ ]:
live_raw = pd.read_csv('live_feature_capture.csv')
print(f'Live capture: {live_raw.shape}')

# Sort chronologically
if 'collected_at' in live_raw.columns:
    live_raw['collected_at'] = pd.to_datetime(live_raw['collected_at'])
    live_raw = live_raw.sort_values('collected_at').reset_index(drop=True)
    duration = live_raw['collected_at'].max() - live_raw['collected_at'].min()
    print(f'Capture duration: {duration}')

# Show old model predictions — FOR REFERENCE ONLY, NOT USED AS LABELS
if 'prediction' in live_raw.columns:
    print(f'\nOld model predictions (DISCARDED, NOT ground truth):')
    print(live_raw['prediction'].value_counts())
    print('=> All 1,416 flows labeled NORMAL because collection was during')
    print('   controlled normal laptop usage, not because of model predictions.')

# Extract only model features
live = live_raw[MODEL_FEATURE_ORDER].copy()
live['label']  = 0
live['source'] = 'windows_benign'
print(f'\nLive feature rows: {len(live)}')

## Cell 5 — Load UNSW-NB15 attack rows

In [ ]:
unsw_raw = pd.read_csv('UNSW_NB15_training-set.csv')
print(f'UNSW shape: {unsw_raw.shape}')
print(f'Label distribution:')
print(unsw_raw['label'].value_counts())

missing = [f for f in MODEL_FEATURE_ORDER if f not in unsw_raw.columns]
if missing:
    raise ValueError(f'Missing features in UNSW: {missing}')

unsw_attacks = unsw_raw[unsw_raw['label']==1][MODEL_FEATURE_ORDER].copy()
unsw_attacks['label']  = 1
unsw_attacks['source'] = 'unsw_attack'
print(f'\nUSW attack rows: {len(unsw_attacks)}')

if 'attack_cat' in unsw_raw.columns:
    print('\nAttack category distribution:')
    print(unsw_raw[unsw_raw['label']==1]['attack_cat'].value_counts())

## Cell 6 — Create leakage-free splits

In [ ]:
from sklearn.model_selection import train_test_split

# ── BENIGN: chronological split ────────────────────────────────────────────
n_live       = len(live)
n_live_train = int(n_live * BENIGN_TRAIN_RATIO)
n_live_test  = n_live - n_live_train

live_train = live.iloc[:n_live_train].copy()
live_test  = live.iloc[n_live_train:].copy()

print('BENIGN split (chronological — first 80% train, last 20% test):')
print(f'  Train: {len(live_train)} flows')
print(f'  Test:  {len(live_test)}  flows')

# ── ATTACK: stratified random split ───────────────────────────────────────
# Stratified by attack_cat if available, else plain random
if 'attack_cat' in unsw_raw.columns:
    attack_cats = unsw_raw[unsw_raw['label']==1]['attack_cat'].values
    attack_train_df, attack_test_df = train_test_split(
        unsw_attacks, test_size=0.20,
        random_state=RANDOM_STATE,
        stratify=attack_cats
    )
    print('\nATTACK split (stratified by attack_cat):')
else:
    attack_train_df, attack_test_df = train_test_split(
        unsw_attacks, test_size=0.20,
        random_state=RANDOM_STATE
    )
    print('\nATTACK split (random):')

print(f'  Train: {len(attack_train_df)} flows')
print(f'  Test:  {len(attack_test_df)}  flows')

# ── Combine ────────────────────────────────────────────────────────────────
train_df = pd.concat([live_train, attack_train_df], ignore_index=True)
test_df  = pd.concat([live_test,  attack_test_df],  ignore_index=True)
train_df = train_df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

n_train_benign = (train_df['label']==0).sum()
n_train_attack = (train_df['label']==1).sum()
n_test_benign  = (test_df['label']==0).sum()
n_test_attack  = (test_df['label']==1).sum()

print(f'\nFINAL SPLITS:')
print(f'  Train: {len(train_df):6d}  ({n_train_benign} benign, {n_train_attack} attack)')
print(f'  Test:  {len(test_df):6d}  ({n_test_benign} benign, {n_test_attack} attack)')

## Cell 7 — DATA LEAKAGE CHECK

In [ ]:
print('='*55)
print('DATA LEAKAGE CHECK')
print('='*55)
print(f'Training Windows benign:  {n_live_train}')
print(f'Test     Windows benign:  {n_live_test}  (last 20%, chronological)')
print(f'Training UNSW attack:     {len(attack_train_df)}')
print(f'Test     UNSW attack:     {len(attack_test_df)}')
print(f'Total train: {len(train_df)}')
print(f'Total test:  {len(test_df)}')

# Check overlap between train and test feature rows
train_feat = train_df[MODEL_FEATURE_ORDER]
test_feat  = test_df[MODEL_FEATURE_ORDER]
train_tuples = set(map(tuple, train_feat.values))
test_tuples  = set(map(tuple, test_feat.values))
overlap = train_tuples & test_tuples
print(f'\nFeature-row overlap between train and test: {len(overlap)}')
if len(overlap) == 0:
    print('PASS: No overlap detected.')
else:
    print(f'WARNING: {len(overlap)} overlapping rows found (may be duplicate flows).')

print('\nPreprocessing will be fitted on training data only.')
print('Test data will be transformed without refitting.')
print('LEAKAGE CHECK COMPLETE')
print('='*55)

## Cell 8 — Handle class imbalance

In [ ]:
ratio = n_train_attack / n_train_benign if n_train_benign > 0 else 1
print(f'Training benign: {n_train_benign}')
print(f'Training attack: {n_train_attack}')
print(f'Attack:Benign ratio: {ratio:.1f}:1')
print()
# scale_pos_weight = n_benign / n_attack
# Tells XGBoost to upweight benign samples
# Prevents model from ignoring the small benign class
scale_pos_weight = n_train_benign / n_train_attack
print(f'scale_pos_weight = {scale_pos_weight:.6f}')
print('XGBoost will upweight each benign sample to compensate for imbalance.')

## Cell 9 — Fit preprocessing on TRAINING DATA ONLY

In [ ]:
from sklearn.preprocessing import LabelEncoder, StandardScaler

def clean_numeric(df):
    df = df.copy()
    for col in NUMERIC:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
        df[col] = df[col].replace([np.inf, -np.inf], 0)
    return df

X_train_raw = clean_numeric(train_df[MODEL_FEATURE_ORDER].copy())
y_train     = train_df['label'].astype(int)
X_test_raw  = clean_numeric(test_df[MODEL_FEATURE_ORDER].copy())
y_test      = test_df['label'].astype(int)
test_source = test_df['source'].values

# Fit LabelEncoders on TRAINING data only
encoders    = {}
X_train_enc = X_train_raw.copy()
X_test_enc  = X_test_raw.copy()

for col in CATEGORICAL:
    le = LabelEncoder()
    train_vals = X_train_raw[col].fillna('-').astype(str).str.strip().str.lower()
    le.fit(train_vals)
    encoders[col] = le
    X_train_enc[col] = le.transform(train_vals)
    test_vals = X_test_raw[col].fillna('-').astype(str).str.strip().str.lower()
    known = set(le.classes_)
    test_vals_safe = test_vals.apply(lambda x: x if x in known else le.classes_[0])
    X_test_enc[col] = le.transform(test_vals_safe)
    print(f'{col} ({len(le.classes_)} classes): {list(le.classes_)}')

# Fit StandardScaler on TRAINING data only
scaler         = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_enc)
X_test_scaled  = scaler.transform(X_test_enc)

print(f'\nTraining matrix: {X_train_scaled.shape}')
print(f'Test matrix:     {X_test_scaled.shape}')
print('Preprocessing fitted on training data ONLY.')

## Cell 10 — Confirm feature order before training

In [ ]:
print('='*55)
print('EXACT FEATURE ORDER USED BY MODEL')
print('='*55)
for i, feat in enumerate(MODEL_FEATURE_ORDER):
    print(f'  {i+1:2d}. {feat}')
print(f'\nTotal: {len(MODEL_FEATURE_ORDER)} features')
print('This order must match live_features.py MODEL_FEATURE_ORDER exactly.')

## Cell 11 — Train XGBoost

In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split as sk_split

# Internal validation split from training data (for monitoring loss)
X_tr, X_vl, y_tr, y_vl = sk_split(
    X_train_scaled, y_train,
    test_size=0.10, random_state=RANDOM_STATE, stratify=y_train
)
print(f'Internal train/val: {len(X_tr)} / {len(X_vl)}')
print(f'Final test set still untouched: {len(X_test_scaled)} rows')
print()
print('Training XGBoost V3...')

model = XGBClassifier(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.08,
    subsample=0.85,
    colsample_bytree=0.85,
    objective='binary:logistic',
    eval_metric='logloss',
    scale_pos_weight=scale_pos_weight,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
model.fit(X_tr, y_tr, eval_set=[(X_vl, y_vl)], verbose=50)
print('Training complete.')

## Cell 12 — Evaluate on held-out test set

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

# First and only time the test set is used
y_pred = model.predict(X_test_scaled)
y_prob = model.predict_proba(X_test_scaled)[:,1]

acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, zero_division=0)
rec  = recall_score(y_test, y_pred, zero_division=0)
f1   = f1_score(y_test, y_pred, zero_division=0)
cm   = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()
fpr  = fp / (fp + tn) if (fp + tn) > 0 else 0
fnr  = fn / (fn + tp) if (fn + tp) > 0 else 0

print('='*60)
print('ARGUS V3 — HELD-OUT TEST RESULTS')
print('='*60)
print(f'Test benign (Windows): {n_test_benign}')
print(f'Test attack (UNSW):    {n_test_attack}')
print()
print(f'Accuracy:   {acc*100:.2f}%')
print(f'Precision:  {prec*100:.2f}%')
print(f'Recall:     {rec*100:.2f}%')
print(f'F1 Score:   {f1*100:.2f}%')
print(f'FPR:        {fpr*100:.2f}%  ({fp} of {n_test_benign} benign misclassified)')
print(f'FNR:        {fnr*100:.2f}%  ({fn} of {n_test_attack} attacks missed)')
print()
print('Confusion Matrix:')
print(f'  TN={tn}  FP={fp}')
print(f'  FN={fn}  TP={tp}')
print()
print(classification_report(
    y_test, y_pred,
    target_names=['Normal','Attack'], digits=4
))

# Windows benign separately
win_mask   = test_source == 'windows_benign'
win_preds  = y_pred[win_mask]
win_probs  = y_prob[win_mask]
n_win_test = win_mask.sum()
win_fp     = (win_preds==1).sum()
win_fpr    = win_fp / n_win_test if n_win_test > 0 else 0

print('='*60)
print('WINDOWS BENIGN TEST FLOWS (284 held-out)')
print('='*60)
print(f'Total:        {n_win_test}')
print(f'Pred NORMAL:  {(win_preds==0).sum()}')
print(f'Pred ATTACK:  {win_fp}')
print(f'FPR:          {win_fpr*100:.2f}%  (FP / actual benign)')
print(f'Avg prob:     {win_probs.mean()*100:.1f}%')
print(f'Median prob:  {np.median(win_probs)*100:.1f}%')

# UNSW attack separately
atk_mask   = test_source == 'unsw_attack'
atk_preds  = y_pred[atk_mask]
atk_probs  = y_prob[atk_mask]
atk_dr     = (atk_preds==1).mean()*100

print()
print('='*60)
print('UNSW ATTACK TEST FLOWS')
print('='*60)
print(f'Total:           {atk_mask.sum()}')
print(f'Detected:        {(atk_preds==1).sum()}')
print(f'Missed:          {(atk_preds==0).sum()}')
print(f'Detection rate:  {atk_dr:.1f}%')
print(f'Avg prob:        {atk_probs.mean()*100:.1f}%')

## Cell 13 — Save all artifacts

In [ ]:
# Save model
model.save_model('xgb_model_v3.json')
print('Saved: xgb_model_v3.json')

with open('scaler_v3.pkl','wb') as f: pickle.dump(scaler, f)
print('Saved: scaler_v3.pkl')

with open('label_encoders_v3.pkl','wb') as f: pickle.dump(encoders, f)
print('Saved: label_encoders_v3.pkl')

# Test predictions CSV
test_out = pd.DataFrame({
    'true_label':         y_test.values,
    'predicted_label':    y_pred,
    'attack_probability': y_prob,
    'source':             test_source,
    'correct':            (y_pred == y_test.values).astype(int),
})
test_out.to_csv('v3_test_predictions.csv', index=False)
print('Saved: v3_test_predictions.csv')

# Config
config = {
    'model': 'ARGUS V3 XGBoost Domain-Adapted',
    'features': MODEL_FEATURE_ORDER,
    'categorical_features': CATEGORICAL,
    'preprocessing': {
        'categorical': 'LabelEncoder fitted on training only',
        'numeric': 'StandardScaler fitted on training only',
    },
    'training_data': {
        'windows_benign_train': int(n_live_train),
        'unsw_attack_train':    int(len(attack_train_df)),
        'total':                int(len(train_df)),
        'scale_pos_weight':     float(scale_pos_weight),
    },
    'test_data': {
        'windows_benign_test': int(n_live_test),
        'unsw_attack_test':    int(len(attack_test_df)),
        'total':               int(len(test_df)),
    },
    'split_strategy': {
        'windows': 'chronological 80/20',
        'unsw_attacks': 'stratified random 80/20 seed=42',
    },
    'threshold': 0.5,
    'metrics': {
        'accuracy':  round(acc*100,2),
        'precision': round(prec*100,2),
        'recall':    round(rec*100,2),
        'f1':        round(f1*100,2),
        'fpr':       round(fpr*100,2),
        'fnr':       round(fnr*100,2),
        'tn': int(tn), 'fp': int(fp),
        'fn': int(fn), 'tp': int(tp),
    },
    'windows_benign_fpr':      round(win_fpr*100,2),
    'unsw_attack_detection':   round(atk_dr,2),
}
with open('model_config_v3.json','w') as f: json.dump(config, f, indent=2)
print('Saved: model_config_v3.json')
print('\nAll 5 artifacts saved.')

## Cell 14 — Download artifacts

In [ ]:
from google.colab import files
for fname in ['xgb_model_v3.json','scaler_v3.pkl',
              'label_encoders_v3.pkl','model_config_v3.json',
              'v3_test_predictions.csv']:
    files.download(fname)
    print(f'Downloaded: {fname}')
print('\nPaste the output of Cell 12 back to Claude.')